# 04_feature_engineering — 候補×特徴量テーブルの作成

`03_candidate_generation.ipynb` で検証した5候補ソース(`candidates.py`に共通化)を使い、
(customer_idx, article_id) ペアごとに user系・item系・user-item交差系・時間系の特徴量とラベルを作る。
特徴量テーブル構築のロジック自体は `features.py` の `build_feature_table()` に共通化してあり、05/06からも再利用する。

## 学習/検証の週構成(改訂: 学習週を2→4週に拡張)

季節性が強く単週バックテストは不安定(hypothesis_driven_design.md 7節)なため、
非重複の5週(学習4週+検証1週)を使う。学習週を増やすのは、直近1〜2週間だけで学習すると
その週固有のクセ(特定商品のスパイクなど)にモデルが過適合しやすいという懸念への対応。

| 用途 | cutoff | ラベル期間 |
|---|---|---|
| 検証(val) | 最終日-7日 | 最終日-7日 の翌日 〜 最終日 |
| 学習(train1) | 最終日-14日 | train1 cutoff の翌日 〜 val cutoff |
| 学習(train2) | 最終日-21日 | train2 cutoff の翌日 〜 train1 cutoff |
| 学習(train3) | 最終日-28日 | train3 cutoff の翌日 〜 train2 cutoff |
| 学習(train4) | 最終日-35日 | train4 cutoff の翌日 〜 train3 cutoff |

valは05でモデル評価専用に使うため学習には混ぜない。全cutoffが2020年8〜9月に収まっており、
COVID期(2020年3〜4月)のレジームシフトは含まれない。

## 対象顧客の絞り込み

全顧客(137万人)×候補60件をcutoffごとに毎回計算すると重いため、
「cutoff以前56日以内に購入実績のある顧客」∪「ラベル期間に購入した顧客」に絞る
(未購入が続く顧客まで含めても学習的価値が薄く、計算コストだけが増えるため)。

## 実行方式についての注記(OOM対策)

5週分を1つのJupyterカーネルで順番に処理する実装は、macOSのjetsamログでPythonプロセスが
OOM killされることを確認した(候補プール・共起テーブル・ALS行列・5週分のDataFrameを
1プロセスに溜め込んだことが原因とみられる)。そのため、**cutoffごとに`build_one_cutoff.py`を
独立したサブプロセスとして呼び出し**、各cutoffの処理が終わるたびにOSにメモリを完全に返却させる
方式に変更している。

In [ ]:
import os
import sys
import glob
import subprocess

import pandas as pd

sys.path.append(os.path.abspath("."))
from common import get_con

FEATURES_DIR = os.path.join("..", "02_output", "features")

con = get_con()
max_dat = con.execute("SELECT MAX(t_dat) FROM trans").fetchone()[0]
con.close()

cutoffs = {
    "val": (max_dat - pd.Timedelta(days=7)).strftime("%Y-%m-%d"),
    "train1": (max_dat - pd.Timedelta(days=14)).strftime("%Y-%m-%d"),
    "train2": (max_dat - pd.Timedelta(days=21)).strftime("%Y-%m-%d"),
    "train3": (max_dat - pd.Timedelta(days=28)).strftime("%Y-%m-%d"),
    "train4": (max_dat - pd.Timedelta(days=35)).strftime("%Y-%m-%d"),
}
print(f"max_dat={max_dat}")
for name, c in cutoffs.items():
    print(f"  {name}: {c}")

## 特徴量

- user系: 購入頻度・直近性・平均単価・価格CV・channel利用比率・週アクティブ数・index_group別購入シェア(H2a/H2b/H2c)
- item系: 総売上・直近7日/28日の売上トレンド・最終販売からの経過日数(死に筋フラグ、H10)・平均価格
- user-item交差系: 当該article/product_codeの過去購入回数・経過日数(H1)・価格乖離(H5/H7)
- 候補ソースのフラグ(from_repeat/from_trend/from_category_color/from_cf/from_cooccurrence、n_sources)
- 時間系: 予測対象週の開始日のday-of-month(給料日効果、H11)

## 5週分をサブプロセスで1つずつ構築

各cutoffの処理を`build_one_cutoff.py`の独立プロセスとして実行し、一時parquetに保存する。

In [ ]:
tmp_paths = {}
for name, c in cutoffs.items():
    tmp_path = os.path.join(FEATURES_DIR, f"_tmp_{name}.parquet")
    tmp_paths[name] = tmp_path
    print(f"=== {name} (cutoff={c}) をサブプロセスで実行 ===")
    result = subprocess.run(
        [sys.executable, "build_one_cutoff.py", "--name", name, "--cutoff", c, "--out", tmp_path],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr[-4000:])
        raise RuntimeError(f"{name} の構築に失敗しました(returncode={result.returncode})")

## 学習4週を結合し、検証週と合わせて保存

一時ファイルはここで読み込んで結合したあとに削除する。

In [ ]:
from datetime import datetime

val_df = pd.read_parquet(tmp_paths["val"])
train_df = pd.concat(
    [pd.read_parquet(tmp_paths[name]) for name in ["train1", "train2", "train3", "train4"]],
    ignore_index=True,
)
print(f"train_df shape={train_df.shape}, val_df shape={val_df.shape}")

## サマリ確認

In [ ]:
summary_rows = []
for name in cutoffs:
    df = val_df if name == "val" else pd.read_parquet(tmp_paths[name])
    summary_rows.append({
        "name": name, "cutoff": cutoffs[name], "n_rows": len(df),
        "n_customers": df.customer_idx.nunique(), "positive_rate": df.label.mean(),
    })
summary = pd.DataFrame(summary_rows)
summary

In [ ]:
print("特徴量一覧:")
for c in train_df.columns:
    print(" -", c, train_df[c].dtype)

print()
print("欠損率(train_df, 上位10):")
print((train_df.isnull().mean().sort_values(ascending=False) * 100).round(2).head(10))

## 保存

`02_output/features/` にタイムスタンプ付きで保存する。05_train_lgbm.ipynb は最新のファイルを読み込む。

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
train_path = os.path.join(FEATURES_DIR, f"train_features_{timestamp}.parquet")
val_path = os.path.join(FEATURES_DIR, f"val_features_{timestamp}.parquet")

train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)

print(f"保存しました: {train_path}")
print(f"保存しました: {val_path}")

for p in tmp_paths.values():
    os.remove(p)
print("一時ファイルを削除しました")

## 次のステップ

- `05_train_lgbm.ipynb`: この特徴量テーブル(5候補ソース×学習4週)でLightGBM Ranker/Classifierを再学習する
- 現行モデル(2週学習・4ソース、バックテストMAP@12=0.0322、Public LB=0.02527)と比較する
- 特徴量重要度で`from_cooccurrence`がどの程度効くか確認する